# Class 2 — Custom GPTs & Assistants
**Week 3: No-Code & Low-Code AI Builders — "The Workflow Canvas"**

### Learning objectives
By the end of this notebook you will be able to:
- Package a system prompt into reusable **instructions** (role, scope, tone, refusal rule)
- Ground a persona's answers in a **knowledge file** and see the difference it makes
- Wire a Python function into the model as a real **tool call** via Groq, the same pattern as a GPT Action
- Spot where custom GPTs go wrong: leaking instructions, knowledge-file bloat, scope creep, untested refusals

Live cells need a `GROQ_API_KEY` (see Setup). Conceptual / writing cells work without one.

Run each cell in order with **Shift+Enter**.

## Setup
**Running in Google Colab:**
1. Get a free key from https://console.groq.com/keys
2. Click the key icon (🔑 Secrets) in the left sidebar
3. Add a secret named `GROQ_API_KEY`, paste your key, toggle **Notebook access** on
4. Run the two setup cells below

**Elsewhere:** set `GROQ_API_KEY` as an environment variable before launching Jupyter.

In [ ]:
!pip install -q groq

In [ ]:
import os

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    print(
        "No API key found \u2014 discussion and writing cells still work.\n"
        "In Colab: add a secret named GROQ_API_KEY via the \U0001F511 Secrets panel and enable notebook access.\n"
        "Elsewhere: set GROQ_API_KEY as an environment variable before launching Jupyter."
    )
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("GROQ_API_KEY loaded \u2014 live cells will work.")

## 1. Instructions: Packaging a Persona
A **custom GPT / Assistant** is a system prompt (instructions) plus optional knowledge and tools, saved so nobody has to retype it. Good instructions cover four things: **role**, **scope & boundaries**, **tone**, and **refusals & escalation**. Below we write one persona, OrderBot, and reuse it for the rest of this notebook.

In [ ]:
def call_llm(messages, model="llama-3.3-70b-versatile", max_tokens=300, temperature=0.4):
    """Send a message list to Groq and return assistant text, or an error string."""
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        return "Error: GROQ_API_KEY is not set."
    try:
        from groq import Groq
        client = Groq(api_key=api_key)
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=max_tokens,
            temperature=temperature,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error calling Groq: {e}"


# "Instructions" -- role, scope, tone, and a refusal rule, all in one system prompt
SUPPORT_BOT_INSTRUCTIONS = (
    "You are OrderBot, a support-triage assistant for an online store.\n"
    "Scope: only answer questions about orders, shipping, and returns.\n"
    "Tone: friendly, concise, no more than 3 sentences.\n"
    "Refusal rule: if asked anything outside that scope, say so and suggest contacting a human agent."
)

messages = [
    {"role": "system", "content": SUPPORT_BOT_INSTRUCTIONS},
    {"role": "user", "content": "Where's my package? It's been 5 days."},
]

if os.environ.get("GROQ_API_KEY"):
    print(call_llm(messages))
else:
    print("Set GROQ_API_KEY, then re-run this cell to see OrderBot reply.")

## 2. Knowledge Files: Grounding the Persona
A **knowledge file** lets the persona quote your actual documents instead of guessing. In the API, that's just extra text appended to the system prompt (or retrieved and inserted before the call). Below we ask the same policy question with and without the "file" attached.

In [ ]:
POLICY_DOC = (
    "Return Policy: Items can be returned within 30 days of delivery for a full refund. "
    "Items must be unused and in original packaging. Refunds are issued to the original "
    "payment method within 5-7 business days."
)

question = "Can I return a shirt I opened and wore once, 20 days after it arrived?"

# without grounding -- the model has to guess
no_knowledge = [
    {"role": "system", "content": SUPPORT_BOT_INSTRUCTIONS},
    {"role": "user", "content": question},
]

# with grounding -- the "knowledge file" is appended to the system prompt
with_knowledge = [
    {"role": "system", "content": SUPPORT_BOT_INSTRUCTIONS + "\n\nKnowledge file (return-policy.txt):\n" + POLICY_DOC},
    {"role": "user", "content": question},
]

if os.environ.get("GROQ_API_KEY"):
    print("--- without knowledge file ---")
    print(call_llm(no_knowledge))
    print("\n--- with knowledge file ---")
    print(call_llm(with_knowledge))
else:
    print("Set GROQ_API_KEY, then re-run this cell to compare grounded vs ungrounded answers.")

## 3. Actions & Tools: From Talking to Doing
An **Action** (ChatGPT) or **tool/function call** (Assistants API, and Groq) lets the persona call a real function and use its result. The model decides *when* to call the tool based on the user's question -- we don't hardcode that decision. Below, `get_order_status` is a stand-in for a real order-lookup API.

In [ ]:
import json

def get_order_status(order_id):
    """Stand-in for a real order-lookup API -- this is the 'Action' the model can call."""
    fake_db = {"4471": "shipped", "9002": "processing", "1188": "delivered"}
    return {"order_id": order_id, "status": fake_db.get(order_id, "not found")}

ORDER_STATUS_TOOL = {
    "type": "function",
    "function": {
        "name": "get_order_status",
        "description": "Look up the shipping status of an order by its order ID.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "The order ID, e.g. '4471'"},
            },
            "required": ["order_id"],
        },
    },
}

def call_with_tools(messages, tools, available_functions, model="llama-3.3-70b-versatile"):
    """One round of tool-calling: model decides whether to call a tool, we execute it, then ask again."""
    from groq import Groq
    client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

    response = client.chat.completions.create(
        model=model, messages=messages, tools=tools, tool_choice="auto", max_tokens=300,
    )
    reply = response.choices[0].message

    if not reply.tool_calls:
        return reply.content

    messages.append(reply)
    for tool_call in reply.tool_calls:
        fn = available_functions[tool_call.function.name]
        args = json.loads(tool_call.function.arguments)
        result = fn(**args)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result),
        })

    follow_up = client.chat.completions.create(model=model, messages=messages, max_tokens=300)
    return follow_up.choices[0].message.content


messages = [
    {"role": "system", "content": SUPPORT_BOT_INSTRUCTIONS},
    {"role": "user", "content": "What's the status of order #4471?"},
]

if os.environ.get("GROQ_API_KEY"):
    print(call_with_tools(messages, [ORDER_STATUS_TOOL], {"get_order_status": get_order_status}))
else:
    print("Set GROQ_API_KEY, then re-run this cell to see OrderBot call the tool and answer.")

## 4. Claude's Equivalent
Claude has no public "GPT store," but the same three building blocks exist under different names: a **Project with custom instructions** (standing persona), **Project knowledge / uploaded files** (grounding documents), and **tool use / function calling via the API** (calling external systems). Same underlying idea, packaged differently by each platform.

### Week 3, Class 2 — closed
A custom GPT/Assistant is instructions + knowledge + tools, packaged once and reused. Instructions cover role, scope, tone, and refusals. Knowledge files ground answers in real documents -- more isn't automatically better. Actions/tools turn "talking" into "doing," and the model itself decides when to reach for one. Class 3 wires that same AI step into a no-code automation platform.

## Challenges
Work through these in order. No solutions are provided — each starter cell has a `# TODO` marking where your code goes. Challenges 1, 3, and 4 need `GROQ_API_KEY`.

### Challenge 1 — Write Instructions for a New Persona
Pick a persona **other than OrderBot** (e.g. a study-buddy tutor, a code reviewer, an internal FAQ bot). Write its instructions covering all four parts: role, scope & boundaries, tone, and a refusal rule. Build a `messages` list with it as the system message plus one user question, call `call_llm`, and print the reply.

**Acceptance criteria:** instructions string covers all four parts; prints a reply from a persona that is not OrderBot.

In [ ]:
# TODO: write a new instructions string (role, scope, tone, refusal rule) for a persona
# other than OrderBot, build a messages list with it + one user question, call call_llm, print the reply

### Challenge 2 — Draft a Knowledge-File Outline
Without calling the API: pick a real topic (your team's onboarding doc, a product FAQ, a course syllabus). Outline what a knowledge file for it would contain — at least **4 section headers**, each with one example fact. Then write 1-2 sentences on what you'd deliberately leave out and why (see Common Pitfalls: knowledge-file bloat).

**Acceptance criteria:** at least 4 section headers with an example fact each, plus your exclusion note.

*(Write your outline here.)*

### Challenge 3 — Simulate a Different Tool Call
Define a new Python function (not `get_order_status`) standing in for a real action — e.g. `check_inventory(sku)`, `get_shipping_estimate(zip_code)`, or `lookup_faq(topic)`. Write its tool schema, then call `call_with_tools` with a user question that should trigger it, and print the final answer.

**Acceptance criteria:** defines a new function + tool schema; prints a reply that used the tool's result.

In [ ]:
# TODO: define a new function + its tool schema (dict) like ORDER_STATUS_TOOL, then call
# call_with_tools(messages, [your_tool], {"your_function_name": your_function}) and print the result

### Challenge 4 — Add a Refusal Rule and Test It
Take your persona from Challenge 1 (or OrderBot) and add one explicit refusal rule if it doesn't already have a sharp one (e.g. "never share another customer's order data", "never give legal or medical advice"). Ask it a trick question designed to cross that line, print the reply, and write one sentence on whether the refusal actually held.

**Acceptance criteria:** prints the model's reply to the trick question plus your one-sentence verdict.

In [ ]:
# TODO: add/sharpen a refusal rule in your instructions, ask a trick question that should trigger it,
# call call_llm, print the reply, then print your one-sentence verdict on whether it refused